In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import GoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv
import os

In [2]:
load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")
model = GoogleGenerativeAI(model="gemini-2.5-flash",google_api_key=api_key)

In [3]:
class BlogState(TypedDict):
    title: str
    outline: str
    content: str

In [4]:
def create_outline(state: BlogState) -> BlogState:
    title = state['title']
    prompt = f'Create a detailed outline for a blog on topic: {title}'
    outline = model.invoke(prompt)
    state['outline'] = outline
    return state   

In [5]:
def create_blog(state: BlogState) -> BlogState:
    title = state['title']
    outline = state['outline']
    prompt = f'Write a blog post based on the following title and outline. Title: {title} Outline: {outline}'
    content = model.invoke(prompt)
    state['content'] = content
    return state

In [6]:
graph = StateGraph(BlogState)

graph.add_node('create_outline',create_outline)
graph.add_node('create_blog',create_blog)

graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', END)

workflow = graph.compile()

In [ ]:
inital_state = {
    'title': "The future of AI"
}

final_state = workflow.invoke(inital_state)
print(final_state)

{'title': 'The future of AI', 'outline': 'Here\'s a detailed outline for a blog post on "The Future of AI," designed to be comprehensive, engaging, and thought-provoking.\n\n---\n\n## Blog Title Options:\n*   **The AI Horizon: Navigating the Future of Artificial Intelligence**\n*   **Beyond Today\'s Bots: What\'s Next for AI?**\n*   **Shaping Tomorrow: The Opportunities and Challenges of AI\'s Evolution**\n*   **The Unfolding Future: A Deep Dive into Artificial Intelligence**\n\n---\n\n### **I. Introduction (Approx. 150-200 words)**\n\n*   **A. Hook:** Start with a captivating statement about AI\'s current impact (e.g., "From ChatGPT\'s eloquent prose to DALL-E\'s stunning visuals, AI is no longer a futuristic dream, but a daily reality.")\n*   **B. Context:** Briefly acknowledge the rapid advancements in AI (Generative AI, LLMs, Computer Vision) and how it has surpassed many expectations.\n*   **C. Thesis Statement:** The blog will explore the multifaceted future of AI, delving into i